# Random first action with success quotas

Use a GPU runtime. This repeats the wider square-band curriculum in the right room with the same six goals, one-macro MPC, and temporal ranking. Each attempt independently starts with a random primitive action with probability `P_RANDOM`. That action is sampled uniformly from the disk of allowed action lengths. MPC then starts from the actual resulting observation. Both groups have the same 100-simulator-action budget.

Each goal in each band gets up to 24 attempts by default, stopping once it has six successes that include an MPC action. A goal that reaches the cap short of quota is reported separately. The fit uses whatever eligible pairs were collected, and retains the previous checkpoint if no new pairs are available. Failed rollouts and the random transition contribute no pairs. The remaining successful MPC transitions use time order with no 0.5-unit progress filter.

The default runs through Band 5, the first previously unsolved band. Set `LAST_BAND=8` to continue across the right room. To resume a run, keep the probability, quota and attempt cap fixed and set `RESUME_RUN_ID` to the printed ID. Raising `LAST_BAND` is supported on resume.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
P_RANDOM=.5
SUCCESS_QUOTA=6            # Successful rollouts with MPC per goal and band
MAX_ATTEMPTS_PER_GOAL=24
LAST_BAND=5                # Includes the previous failure band. Set to 8 to cover the right room.
PLOT_GOALS=[1]              # Collection goal indices for the trajectory panels
PLOT_BANDS=[2,3,4,5]
DRAW_COST_MAPS=True
HEATMAP_GOALS=[0,15]        # Evaluation goal indices for the raw vs adapted maps
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_random_first_half_square_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-random-first')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='a09d0a4215d89d5480a1168ba0fd319ceeba3d70'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-one-macro-adapter-wide-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_curriculum',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',MAX_ATTEMPTS_PER_GOAL,'--labels','temporal','--sampling','half-square','--planning-horizon',1,
          '--actions-per-plan',1,'--wide-square-bands',
          '--success-quota',SUCCESS_QUOTA,'--random-first-probability',P_RANDOM]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Collect and fit

The current checkpoint stays fixed during collection of each band. All completed episodes are journaled. Failed runs store positions for diagnosis and successful runs also save the frames needed for fitting.

In [ ]:
assert 1 <= LAST_BAND <= 8
for band in range(1,LAST_BAND+1):
    run_band(band)

## Compare the random-first and MPC-first groups

Band entry means the ball reached the previous square while still in the right room. The first-action and five-action columns count entry by the same simulator step in both groups. Band 1 has no previous square. Positive distance changes mean the ball moved farther from the goal. The near-edge subset keeps starts whose square distance to the boundary is within one allowed action length.

Check denominators for each goal and arm. Arms are randomly assigned across starts, so the rates are group comparisons with quota-based stopping.

In [ ]:
from IPython.display import Image, display
import numpy as np
import pandas as pd

def read_json(path):
    return json.loads(path.read_text())

manifest=read_json(OUTPUT/'experiment.json')['manifest']
stages={}
for band in range(1,LAST_BAND+1):
    folder=OUTPUT/f'band_{band}'
    if (folder/'complete.json').is_file():
        stages[band]=read_json(folder/'complete.json')['collection']
    elif (folder/'collection.json').is_file():
        stages[band]=read_json(folder/'collection.json')
if not stages:
    raise RuntimeError('Run at least one collection band first.')

rows=[]
for band, episodes in stages.items():
    inner=manifest['bands'][band-1][0]
    for row in episodes:
        case=row['case']
        diag=row['exploration']
        gap=max(abs(x-g) for x,g in zip(case['start'],case['goal']))-inner
        rows.append(dict(band=band,goal=case['goal_index'],arm=diag['arm'],success=row['success'],
            mpc_success=row['success'] and row['steps']>diag['prefix_steps'],pairs=row['retained_pairs'],
            steps=row['steps'],entered_first=diag['entered_by_first_action'] if inner else np.nan,
            entered_five=diag['entered_by_five_actions'] if inner else np.nan,
            entered=diag['first_entry_step'] is not None if inner else np.nan,
            near_edge=bool(inner and gap<=manifest['planning']['max_action_length']),
            delta_first=diag['first_distance_change'],delta_five=diag['first_five_distance_change'],
            ended_farther=diag['final_distance_change']>0))
data=pd.DataFrame(rows)

def summarize(grouped):
    return grouped.agg(attempts=('success','size'),successes=('success','sum'),
        success_rate=('success','mean'),entry_step1=('entered_first','mean'),
        entry_step5=('entered_five','mean'),entry_ever=('entered','mean'),
        median_delta_step1=('delta_first','median'),median_delta_step5=('delta_five','median'),
        ended_farther_rate=('ended_farther','mean'),retained_pairs=('pairs','sum')).round(3)

print('By band and policy')
display(summarize(data.groupby(['band','arm'])))
print('Within one action length of the previous square')
display(summarize(data[data.near_edge].groupby(['band','arm'])))
print('By band, goal and policy')
per_goal=summarize(data.groupby(['band','goal','arm']))
display(per_goal)
quota=(data.groupby(['band','goal']).agg(attempts=('success','size'),
        mpc_successes=('mpc_success','sum'),retained_pairs=('pairs','sum')))
quota['quota_met']=quota.mpc_successes>=manifest['exploration']['success_quota']
print('Success quota by goal')
display(quota)
per_goal.to_csv(OUTPUT/'exploration_by_goal.csv')
quota.to_csv(OUTPUT/'exploration_quotas.csv')

## See where the ball went

Each figure compares the two policy groups for the same goal and band. Green lines show successful trajectories. Red traces show the first five actions of failures, and dashed red segments connect their starts and ends. Amber marks the random action, white dots mark retained training observations, and the inner square is the boundary we want to cross.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt

plot_dir=OUTPUT/'exploration_figures'
plot_dir.mkdir(exist_ok=True)
for band in PLOT_BANDS:
    for index in PLOT_GOALS:
        episodes=[r for r in stages.get(band,[]) if r['case']['goal_index']==index]
        if not episodes:
            continue
        gx,gy=episodes[0]['case']['goal']
        inner,outer=manifest['bands'][band-1]
        fig,axes=plt.subplots(1,2,figsize=(10,7.2),layout='constrained')
        for ax,arm in zip(axes,('MPC first','random first')):
            group=[r for r in episodes if r['exploration']['arm']==arm]
            for row in sorted(group,key=lambda r:r['success']):
                diag=row['exploration']
                xy=np.asarray(diag['positions'])
                color='#208445' if row['success'] else '#c83d46'
                if row['success']:
                    ax.plot(*xy.T,color=color,alpha=.68)
                    retained=np.asarray(row['retained_positions']).reshape(-1,2)
                    if len(retained):
                        ax.scatter(*retained.T,s=13,c='white',edgecolors=color,linewidths=.65,zorder=4)
                else:
                    ax.plot(*xy[:6].T,color=color,alpha=.52,linewidth=1.1)
                    ax.plot(*xy[[0,-1]].T,color=color,linestyle='--',alpha=.35)
                if diag['prefix_steps']:
                    ax.plot(*xy[:2].T,color='#e69316',linewidth=2.3,alpha=.9,zorder=3)
                ax.scatter(*xy[0],s=32,facecolors='none',edgecolors=color,zorder=5)
                ax.scatter(*xy[-1],s=30,color=color,marker='x',zorder=5)
            for radius,style in ((inner,'-'),(outer,'--')):
                if radius:
                    ax.add_patch(Rectangle((gx-radius,gy-radius),2*radius,2*radius,
                                           fill=False,edgecolor='#555',linestyle=style,linewidth=1.1))
            ax.scatter(gx,gy,marker='*',s=120,c='black',zorder=6)
            successes=sum(r['success'] for r in group)
            entries=sum(r['exploration']['first_entry_step'] is not None for r in group)
            outcome=f'{successes}/{len(group)} success' if group else 'no attempts'
            if group and inner:
                outcome+=f' · {entries}/{len(group)} entered'
            ax.set(title=f'{arm} · {outcome}',xlim=(35,60),ylim=(4,60),aspect='equal',
                   xlabel='Ball x',ylabel='Ball y')
        fig.suptitle(f'Collection goal {index} ({gx:.2f}, {gy:.2f}) · Band {band}')
        fig.legend(handles=[
            Line2D([],[],color='#208445',label='Success'),
            Line2D([],[],color='#c83d46',label='Failed first five'),
            Line2D([],[],color='#c83d46',ls='--',label='Failed start to end'),
            Line2D([],[],color='#e69316',label='Random first action'),
            Line2D([],[],marker='o',mfc='white',mec='#208445',ls='none',label='Training observation'),
        ],loc='upper center',bbox_to_anchor=(.5,-.005),ncol=5,frameon=False)
        path=plot_dir/f'goal_{index}_band_{band}.png'
        fig.savefig(path,dpi=135,bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))

## Raw and adapted cost maps

Optional. The maps show the checkpoints after each completed band and cover the right room through x = 60. Orange outlines mark the band of each checkpoint. The numerical summary uses the same held-out evaluation goals and positions as the wider one-macro experiment.

In [ ]:
if DRAW_COST_MAPS:
    last_completed=max(band for band in stages if (OUTPUT/f'band_{band}'/'complete.json').is_file())
    shell(command('compare',last_completed),cwd=REPO)
    report=read_json(OUTPUT/f'band_{last_completed}_comparison.json')
    summary=pd.DataFrame([
        dict(cost=method,correct_order=np.mean([r['adapted_rank']['correct_fraction'] for r in rows[1:]]),
             uphill=np.mean([r['heldout_directional']['adapted_up_fraction'] for r in rows[1:]]))
        for method,rows in report['methods'].items()
    ])
    display(summary.round(3))

    manifest = json.loads((OUTPUT / 'experiment.json').read_text())['manifest']
    PLOT_DIR = OUTPUT / 'heatmaps'

    from IPython.display import Image, display
    from matplotlib import patheffects
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle
    import matplotlib.pyplot as plt
    import numpy as np

    grid = report['grid']
    shape = (len(grid['ys']), len(grid['xs']))
    methods = list(report['methods'])
    bands = manifest['bands']
    collection_goals = {tuple(case['goal']) for stage in manifest['stages'] for case in stage}
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    for goal_id in HEATMAP_GOALS:
        goal = tuple(report['methods']['raw'][goal_id]['goal'])
        gx, gy = goal
        cols = 5
        rows = (len(methods) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(17, 4.8 * rows),
                                 squeeze=False, constrained_layout=True)
        for ax in axes.flat[len(methods):]:
            ax.set_visible(False)
        for ax, method in zip(axes.flat, methods):
            row = report['methods'][method][goal_id]
            costs = np.asarray(row['adapted_costs'])[grid['point_ids']].reshape(shape)
            mesh = ax.pcolormesh(grid['xs'], grid['ys'], costs, shading='nearest', cmap='viridis',
                                 vmin=0, vmax=max(float(costs.max()), 1e-9))
            active = ()
            title = {'raw': 'Raw MSE', 'handcrafted': 'Hand-crafted'}.get(method, method)
            if method.startswith('band_'):
                number = int(method.split('_')[1])
                active = bands[number - 1]
                title = f'Band {number} · {active[0]}–{active[1]}'
                if report['checkpoints'][method]['updates'] == 0:
                    title += '\nCarried forward (0 updates)'
            for _, radius in bands:
                highlight = radius in active
                width = 1.6 if highlight else 0.8
                outline = Rectangle((gx - radius, gy - radius), 2 * radius, 2 * radius,
                                    fill=False, edgecolor='#ffaf36' if highlight else 'white',
                                    linestyle='-' if highlight else '--', linewidth=width,
                                    zorder=4 if highlight else 3,
                                    path_effects=[patheffects.withStroke(linewidth=width + 0.9,
                                                                          foreground='#282838')])
                ax.add_patch(outline)
            ax.scatter(*goal, marker='*', s=110, c='black', edgecolor='white', zorder=5)
            ax.set(title=title, xlabel='Ball x', xlim=(35, 60), ylim=(4, 60), aspect='equal')
            fig.colorbar(mesh, ax=ax, shrink=.6, label='Latent MSE')
        for ax in axes[:, 0]:
            ax.set_ylabel('Ball y')
        label = 'Collection start bands' if goal in collection_goals else 'Held-out goal · reference bands'
        fig.suptitle(f'Goal {goal_id} {goal} · {label}')
        fig.legend(handles=[Line2D([], [], color='#4e4e5e', linestyle='--', label='Square band boundaries'),
                            Line2D([], [], color='#d57900', label='Inner and outer edges of panel band')],
                   loc='upper center', bbox_to_anchor=(.5, -.005), ncol=2, frameon=False)
        path = PLOT_DIR / f'goal_{goal_id}_bands.png'
        fig.savefig(path, dpi=130, bbox_inches='tight')
        plt.close(fig)
        display(Image(filename=str(path)))